# NB_patient_360
Builds `adb_caresync.gold.patient_360` per STTM *Silver to Gold v1.0* (sheet `patient_360`).

* **Grain:** one row per patient
* **Driving table:** `adb_caresync.silver.patients`
* **Joins (LEFT):** `hospitals` on `registered_hospital_id`, `insurance_providers` on `insurance_provider_id`, `lab_results` aggregated by `patient_id`
* **Filters:** none (all patients)
* **Load:** FULL rebuild via `CREATE OR REPLACE TABLE ... AS SELECT`

Source note: `silver.patients.date_of_birth` is a TIMESTAMP (all midnight); cast to DATE per STTM.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (STTM Silver to Gold v1.0)'
AS
WITH lab_agg AS (
  -- Lab results aggregated to patient grain
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,                                                                          -- PK
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                          AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT)   AS age,
  p.gender,
  p.city,                                                                                -- patient's own city
  p.state,                                                                               -- patient's own state
  p.registered_hospital_id                                                               AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                                                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                                        AS critical_test_count,
  l.last_test_date                                                                       -- NULL if no tests
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
source_table = "adb_caresync.silver.patients"
target_table = "adb_caresync.gold.patient_360"

record_count = spark.table(source_table).count()
target_count = spark.table(target_table).count()
print(f"Step 6: Load finished. Source records processed: {record_count}, total records in {target_table}: {target_count}")
dbutils.notebook.exit(str(record_count))